In [ ]:
# from rag_pipeline import generate_with_hf
from dotenv import load_dotenv
import os
from huggingface_hub import InferenceClient

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")
HF_MODEL = os.getenv("HF_MODEL")


def generate_with_hf(prompt: str, max_new_tokens: int = 256) -> str:
    if not HF_TOKEN:
        return "[HF_TOKEN not configured]"

    client = InferenceClient(api_key=os.environ["HF_TOKEN"])
    # use text_generation endpoint
    completion = client.text_generation(model=HF_MODEL, prompt=prompt, max_new_tokens=max_new_tokens)
    # parse common shapes
    if isinstance(resp, list) and len(resp) > 0 and isinstance(resp[0], dict):
        return resp[0].get("generated_text") or str(resp[0])
    if isinstance(resp, dict):
        return resp.get("generated_text") or str(resp)
    return str(resp)


import os
from huggingface_hub import InferenceClient

def generate_with_hf(prompt: str, max_new_tokens: int = 256) -> str:
    
client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM3-3B:hf-inference",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

############
prompt = "Explain what DocQuery is in one sentence."

print("Calling HF API...")

resp = generate_with_hf(prompt)

print("HF RESPONSE:", resp)

In [6]:
import os
from huggingface_hub import InferenceClient

def generate_with_hf(prompt: str, max_new_tokens: int = 256) -> str:
    
    client = InferenceClient(
        api_key=os.environ["HF_TOKEN"],
    )

    completion = client.chat.completions.create(
        # model="HuggingFaceTB/SmolLM3-3B:hf-inference",
        model = HF_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens = max_new_tokens
         )

    return completion.choices[0].message

q = "What is the capital of France?"
resp = generate_with_hf(q)
resp

ChatCompletionOutputMessage(role='assistant', content="<think>\nOkay, the user is asking for the capital of France. Let me make sure I remember correctly. I think it's Paris. But wait, maybe I should double-check to be sure. Sometimes there are cities that are sometimes mistaken for capitals, like Lyon or Marseille, but those are not the capitals. The capital of France is definitely Paris. I recall that Paris is a major city in France and has been the capital for a long time. Let me think if there's any other city that could be confused with it. No, I don't think so. The capital is Paris. I should also mention that it's a well-known city with famous landmarks like the Eiffel Tower and the Louvre. That might help the user if they're interested in more information. But the main answer is Paris. I should keep it simple and straightforward since the question is direct.\n</think>\n\nThe capital of France is **Paris**. It is a major cultural, economic, and political hub in the country and is

In [10]:
import os
import re
from huggingface_hub import InferenceClient

# HF_MODEL = os.environ.get("HF_MODEL", "HuggingFaceTB/SmolLM3-3B:hf-inference")
# HF_TOKEN = os.environ["HF_TOKEN"]

HF_MODEL = os.environ.get("HF_MODEL")
HF_TOKEN = os.environ["HF_TOKEN"]

In [ ]:
def clean_reasoning(text: str) -> str:
    """Remove <think> ... </think> blocks if present."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def generate_with_hf(
                system_prompt: str,
                prompt: str,
                max_new_tokens: int = 256,
                strip_reasoning: bool = True
            ) -> str:
    """
    Generate text using a HuggingFace Inference Client in chat mode.
    Returns only the assistant's content (string).
    """
    
    client = InferenceClient(api_key=HF_TOKEN)

    completion = client.chat.completions.create(
        model=HF_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens
    )

    msg = completion.choices[0].message
    content = msg.content if hasattr(msg, "content") else str(msg)

    if strip_reasoning:
        content = clean_reasoning(content)

    return content


# sys_prompt = f"""You are DocQuery, a helpful document assistant.
# Use ONLY the context below to answer the question. If the answer cannot be found in the context, respond with "I don't know".
# """

sys_prompt = f"""You are a helpful assistant.
Use ONLY the context below to answer the question. If the answer cannot be found in the context, respond with "I don't know".

Just provide anser to the asked question only, no need to add anything extra information
"""

q = "What is the capital of France?"
resp = generate_with_hf(sys_prompt, q)
resp

'Paris'

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# 1. Define the Graph State
class AgentState(TypedDict):
    messages: str

# 2. Define the Nodes
def greet_user(state: AgentState):
    """Node 1: Adds a greeting message to the state."""
    print("---EXECUTING GREET_USER NODE---")
    mesg = "Hello!"
    if len(state["messages"]) > 0:
        state["messages"] = " ".join(state["messages"] + mesg)
    else:
        state["messages"] = mesg
    print(state["messages"])
    return state

def get_user_name(state: AgentState):
    """Node 2: Simulates getting user input and adds it to the state."""
    print("---EXECUTING GET_USER_NAME NODE---")
    state["messages"] = "My name is LangGraph User."
    combined_message = " ".join(state["messages"])
    print(state["messages"])
    return state

def combine_messages(state: AgentState):
    """Node 3: Combines all messages in the state into a final output."""
    print("---EXECUTING COMBINE_MESSAGES NODE---")
    # combined_message = "combined"
    combined_message = " ".join(state["messages"])
    state["messages"] = [combined_message]
    print(state["messages"])
    return state

# 3. Build the Graph
builder = StateGraph(AgentState)

# Add nodes to the graph
builder.add_node("greet_user", greet_user)
builder.add_node("get_user_name", get_user_name)
builder.add_node("combine_messages", combine_messages)

# Define the edges (flow of execution)
builder.add_edge(START, "greet_user")
builder.add_edge("greet_user", "get_user_name")
builder.add_edge("get_user_name", "combine_messages")
builder.add_edge("combine_messages", END)

# Compile the graph
app = builder.compile(checkpointer = MemorySaver())

# 4. Invoke the Graph
initial_state = {"messages": ""}
# final_state = app.invoke(initial_state)

# graph = get_rag_graph()a
final_state = app.invoke(
    initial_state,
    return_intermediate_steps=False,
    config={
        "configurable": {
            "thread_id": "2",       # memory thread
            "checkpoint_ns": "rag_memory", # memory namespace
        }
    }
)

print("\n---FINAL STATE---")
print(final_state)

---EXECUTING GREET_USER NODE---
Hello!
---EXECUTING GET_USER_NAME NODE---
My name is LangGraph User.
---EXECUTING COMBINE_MESSAGES NODE---
['M y   n a m e   i s   L a n g G r a p h   U s e r .']

---FINAL STATE---
{'messages': ['M y   n a m e   i s   L a n g G r a p h   U s e r .']}
